# Exercício 0: Configurando o acesso à AWS

Antes de chamar o Amazon Bedrock com o `boto3`, precisamos de uma identidade na AWS com as permissões certas e das credenciais dela salvas na sua máquina.

Neste notebook vamos, passo a passo:

1. Instalar a AWS CLI e o `boto3`
2. Criar uma política com as permissões extras dos exercícios (Lambda e S3)
3. Criar um grupo IAM com as políticas do workshop
4. Criar um usuário IAM dentro desse grupo
5. Gerar uma access key para o usuário
6. Salvar as credenciais em um perfil com `aws configure`
7. Testar se tudo funciona

> **Sobre segurança:** a AWS recomenda usar o IAM Identity Center (SSO, com credenciais temporárias) em vez de usuários IAM com chaves de longo prazo. Para um workshop, um usuário IAM com access key é o caminho mais simples. Só não use esse usuário em produção nem com dados reais, e apague a access key ao final do workshop (veja a última seção).

Documentação de referência:
- [Authenticating using IAM user credentials for the AWS CLI](https://docs.aws.amazon.com/cli/latest/userguide/cli-authentication-user.html)
- [Create an IAM user in your AWS account](https://docs.aws.amazon.com/IAM/latest/UserGuide/id_users_create.html)

# 1. Instalando a AWS CLI e o boto3

A AWS CLI é a ferramenta de linha de comando da AWS. É ela que fornece o comando `aws configure`, que grava as credenciais usadas também pelo `boto3`.

Instale a **AWS CLI v2** pelo terminal ([guia oficial](https://docs.aws.amazon.com/cli/latest/userguide/getting-started-install.html)):

**Windows** (PowerShell ou Prompt de Comando):
```
msiexec.exe /i https://awscli.amazonaws.com/AWSCLIV2.msi
```

**macOS:**
```
curl "https://awscli.amazonaws.com/AWSCLIV2.pkg" -o "AWSCLIV2.pkg"
sudo installer -pkg AWSCLIV2.pkg -target /
```

**Linux (x86_64):**
```
curl "https://awscli.amazonaws.com/awscli-exe-linux-x86_64.zip" -o "awscliv2.zip"
unzip awscliv2.zip
sudo ./aws/install
```

Depois de instalar, feche e abra o terminal (e o Jupyter) de novo e confira a versão:

In [ ]:
!aws --version

In [ ]:
%pip install --upgrade boto3

# 2. Quais permissões os exercícios precisam?

Os notebooks do workshop usam estes serviços:

| Notebook | O que faz | Permissões |
| --- | --- | --- |
| `exercise1-text1`, `text-stream`, `converse` | Invoca modelos (Nova, Llama) | `bedrock:InvokeModel`, `bedrock:InvokeModelWithResponseStream` |
| `promptManagement-*`, `exercise5-promptManagement` | Cria, versiona, otimiza e executa prompts | `bedrock:CreatePrompt`, `GetPrompt`, `ListPrompts`, `UpdatePrompt`, `CreatePromptVersion`, `OptimizePrompt`, `RenderPrompt` |
| `guardrail` | Aplica um guardrail | `bedrock:ApplyGuardrail`, `bedrock:GetGuardrail` |
| `prompt-routing` | Cria um prompt router | `bedrock:CreatePromptRouter` |
| `exercise1-video` | Gera vídeo com o Nova Reel (assíncrono) | `bedrock:StartAsyncInvoke`, `GetAsyncInvoke` + `s3:PutObject` no bucket de saída |
| `converseToolUse`, `exercise5-lambda` | Chama funções Lambda como ferramentas | `lambda:InvokeFunction` |

Vamos usar duas políticas:

- **`AmazonBedrockFullAccess`** (gerenciada pela AWS): cobre tudo do Bedrock e também as permissões do AWS Marketplace (`aws-marketplace:Subscribe`, ...) necessárias para liberar modelos de terceiros, como o Llama, na primeira chamada. Ela **não** inclui Lambda nem S3.
- **`WorkshopGenAIExtras`** (criada por nós): libera só o que falta, ou seja, invocar as duas funções Lambda e gravar/ler no bucket dos vídeos.

> **Acesso aos modelos:** nas regiões comerciais não existe mais a página "Model access" para pedir acesso. Os modelos são liberados automaticamente na primeira chamada. A exceção são os modelos da Anthropic, que exigem preencher uma vez o formulário de caso de uso (não usamos Anthropic neste workshop). Veja [Access Amazon Bedrock foundation models](https://docs.aws.amazon.com/bedrock/latest/userguide/model-access.html).

# 3. Criando a política `WorkshopGenAIExtras`

Os passos de 3 a 5 são feitos no **Console da AWS**, logado com um usuário que tenha permissão de administrador do IAM. Evite usar o usuário root.

Antes de começar, anote:
- O **nome do bucket S3** onde os vídeos do Nova Reel serão salvos. Se ainda não existir, crie em **S3 > Create bucket**, na região `us-east-1`. O nome precisa ser único no mundo, ex.: `workshop-genai-videos-seunome`.
- Os nomes das funções Lambda usadas como ferramentas: `weather-tool` e `math-function`. As funções precisam existir em `us-east-1` na mesma conta.

Passo a passo:

1. Abra o [console do IAM](https://console.aws.amazon.com/iam/)
2. No menu lateral, clique em **Policies** e depois em **Create policy**
3. Em **Policy editor**, troque de **Visual** para **JSON**
4. Apague o conteúdo e cole o JSON gerado pela célula abaixo (ajuste `BUCKET_NAME` antes de executar)
5. Clique em **Next**
6. Em **Policy name**, digite `WorkshopGenAIExtras` e, se quiser, uma descrição
7. Clique em **Create policy**

In [ ]:
import json

BUCKET_NAME = "workshop-genai-videos-seunome"  # << troque pelo nome do seu bucket
REGION = "us-east-1"

workshop_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Sid": "InvocarFerramentasLambda",
            "Effect": "Allow",
            "Action": "lambda:InvokeFunction",
            "Resource": [
                f"arn:aws:lambda:{REGION}:*:function:weather-tool",
                f"arn:aws:lambda:{REGION}:*:function:math-function"
            ]
        },
        {
            "Sid": "ListarBucketDeVideos",
            "Effect": "Allow",
            "Action": "s3:ListBucket",
            "Resource": f"arn:aws:s3:::{BUCKET_NAME}"
        },
        {
            "Sid": "GravarELerVideos",
            "Effect": "Allow",
            "Action": ["s3:PutObject", "s3:GetObject"],
            "Resource": f"arn:aws:s3:::{BUCKET_NAME}/*"
        }
    ]
}

print(json.dumps(workshop_policy, indent=4))

# 4. Criando o grupo IAM do workshop

A AWS recomenda dar permissões a **grupos** e colocar os usuários dentro deles, em vez de anexar políticas direto em cada usuário. Assim, para vários participantes basta adicionar todos ao mesmo grupo.

1. No console do IAM, clique em **User groups** e depois em **Create group**
2. Em **User group name**, digite `workshop-genai`
3. Em **Attach permissions policies**, busque e marque:
   - `AmazonBedrockFullAccess`
   - `WorkshopGenAIExtras` (dica: mude o filtro para **Customer managed** para achar mais rápido)
4. Clique em **Create user group**

# 5. Criando o usuário IAM

1. No console do IAM, clique em **Users** e depois em **Create user**
2. Na página **Specify user details**:
   - Em **User name**, digite um nome, ex.: `workshop-genai-seunome`
   - **Não** marque **Provide user access to the AWS Management Console**. O usuário só vai acessar a AWS pelo código, então não precisa de senha. Se quiser que ele também use o console, marque a opção e defina a senha
   - Clique em **Next**
3. Na página **Set permissions**:
   - Escolha **Add user to group**
   - Marque o grupo `workshop-genai`
   - Clique em **Next**
4. Na página **Review and create**, confira o nome e o grupo e clique em **Create user**

# 6. Gerando a access key

A access key é o par de credenciais que a CLI e o `boto3` usam para se identificar na AWS: um **Access key ID** (começa com `AKIA`) e uma **Secret access key**.

1. Em **IAM > Users**, clique no usuário que você acabou de criar
2. Abra a aba **Security credentials**
3. Na seção **Access keys**, clique em **Create access key**
4. Em **Use case**, escolha **Command Line Interface (CLI)**. O console vai sugerir alternativas como o IAM Identity Center. Marque a caixa de confirmação (*I understand the above recommendation...*) e clique em **Next**
5. Em **Description tag value** (opcional), digite algo como `workshop genai` e clique em **Create access key**
6. Na página **Retrieve access keys**, clique em **Download .csv file** ou em **Show** para ver a secret
7. Clique em **Done**

> **Atenção:** a secret access key **só aparece uma vez**. Se você perder, precisa apagar a chave e criar outra. Nunca cole essa chave em um notebook, em um repositório git ou em um chat.

# 7. Salvando as credenciais com `aws configure`

Agora vamos gravar a access key na sua máquina. O `aws configure` é interativo, então execute-o em um **terminal** (não em uma célula do notebook, onde ele travaria esperando a digitação).

Vamos criar um perfil com nome, `workshop`, para não sobrescrever nenhuma credencial que você já tenha:

```
$ aws configure --profile workshop
AWS Access Key ID [None]: AKIA................
AWS Secret Access Key [None]: ........................................
Default region name [None]: us-east-1
Default output format [None]: json
```

O que cada pergunta significa:

| Pergunta | O que responder |
| --- | --- |
| **AWS Access Key ID** | O Access key ID do passo 6 |
| **AWS Secret Access Key** | A secret do passo 6 (o terminal mostra o texto colado) |
| **Default region name** | `us-east-1`, a região usada em todos os exercícios |
| **Default output format** | `json` |

> Se este for o único perfil da máquina, você pode rodar só `aws configure` (sem `--profile`). Isso grava no perfil `default`, que o `boto3` usa automaticamente.

**Alternativa:** importar o `.csv` baixado no passo 6. O nome do perfil criado será o valor da coluna *User Name*, que precisa ser adicionada ao arquivo antes da importação:

```
$ aws configure import --csv file://caminho/para/credentials.csv
```

## Onde as credenciais ficam salvas

O comando grava dois arquivos na pasta `.aws` do seu usuário (`~/.aws` no Linux/macOS, `%USERPROFILE%\.aws` no Windows):

**`credentials`**, com as chaves:
```
[workshop]
aws_access_key_id = AKIA................
aws_secret_access_key = ........................................
```

**`config`**, com as configurações:
```
[profile workshop]
region = us-east-1
output = json
```

# 8. Usando o perfil

Os outros notebooks criam os clientes sem passar perfil, por exemplo `boto3.client("bedrock-runtime", region_name="us-east-1")`. Nesse caso o `boto3` procura credenciais nesta ordem e usa a primeira que encontrar:

1. Parâmetros passados no código (`boto3.Session(profile_name=...)`, `aws_access_key_id=...`)
2. Variáveis de ambiente (`AWS_PROFILE`, `AWS_ACCESS_KEY_ID`, `AWS_SECRET_ACCESS_KEY`)
3. O perfil `default` em `~/.aws/credentials` e `~/.aws/config`
4. Role de container/instância, quando o código roda dentro da AWS

Se você criou o perfil `workshop`, a forma mais simples de fazer **todos** os notebooks usarem esse perfil é definir a variável de ambiente `AWS_PROFILE` **antes** de criar os clientes. Execute a célula abaixo no início de cada notebook, ou defina a variável no terminal antes de abrir o Jupyter:

- Windows (PowerShell): `$env:AWS_PROFILE = "workshop"`
- Linux/macOS: `export AWS_PROFILE=workshop`

In [ ]:
import os

os.environ["AWS_PROFILE"] = "workshop"  # use "default" se você rodou aws configure sem --profile

# 9. Testando a configuração

Primeiro pela CLI, conferindo de onde vem cada valor e qual identidade está sendo usada:

In [ ]:
!aws configure list --profile workshop

In [ ]:
!aws sts get-caller-identity --profile workshop

Agora pelo `boto3`. O campo `Arn` deve mostrar o usuário criado no passo 5:

In [ ]:
import boto3
import json
from IPython.display import JSON

In [ ]:
sts = boto3.client(service_name="sts")
JSON(sts.get_caller_identity(), expanded=True)

Por fim, uma checagem rápida de cada permissão usada no workshop. Ela só faz chamadas de leitura e uma chamada barata ao Nova Micro:

In [ ]:
from botocore.exceptions import ClientError

REGION = "us-east-1"
BUCKET_NAME = "workshop-genai-videos-seunome"  # << o mesmo bucket do passo 3

def checar(nome, funcao):
    try:
        funcao()
        print(f"✅ {nome}")
    except ClientError as e:
        print(f"❌ {nome}: {e.response['Error']['Code']} - {e.response['Error']['Message']}")

bedrock = boto3.client(service_name="bedrock", region_name=REGION)
bedrock_runtime = boto3.client(service_name="bedrock-runtime", region_name=REGION)
bedrock_agent = boto3.client(service_name="bedrock-agent", region_name=REGION)
s3 = boto3.client(service_name="s3", region_name=REGION)

checar("Bedrock: listar modelos", lambda: bedrock.list_foundation_models(byProvider="Amazon"))
checar("Bedrock: invocar modelo (Converse)", lambda: bedrock_runtime.converse(
    modelId="amazon.nova-micro-v1:0",
    messages=[{"role": "user", "content": [{"text": "Responda apenas: ok"}]}],
    inferenceConfig={"maxTokens": 10}
))
checar("Bedrock: Prompt Management", lambda: bedrock_agent.list_prompts(maxResults=1))
checar("Bedrock: Guardrails", lambda: bedrock.list_guardrails(maxResults=1))
checar(f"S3: acessar o bucket {BUCKET_NAME}", lambda: s3.list_objects_v2(Bucket=BUCKET_NAME, MaxKeys=1))

# 10. Primeira chamada ao Amazon Bedrock

In [ ]:
MODEL_ID = "amazon.nova-micro-v1:0"

response = bedrock_runtime.converse(
    modelId=MODEL_ID,
    messages=[{"role": "user", "content": [{"text": "Dê as boas-vindas aos participantes do workshop em uma frase"}]}],
    inferenceConfig={"maxTokens": 100}
)

print(response["output"]["message"]["content"][0]["text"])

Se você recebeu uma resposta, está pronto para os próximos exercícios.

## Erros comuns

| Erro | Causa provável |
| --- | --- |
| `NoCredentialsError` | Nenhuma credencial encontrada. Rode o `aws configure` de novo ou confira o `AWS_PROFILE` |
| `ProfileNotFound` | O perfil não existe. Confira o nome com `aws configure list-profiles` |
| `UnrecognizedClientException` / `InvalidClientTokenId` | Access key digitada errada, desativada ou apagada |
| `SignatureDoesNotMatch` | Secret access key errada (geralmente um caractere faltando ao colar) |
| `AccessDeniedException` | O usuário não está no grupo `workshop-genai`, a política não foi anexada, ou o modelo ainda está sendo liberado. Na primeira chamada de modelos de terceiros, a liberação pode levar alguns minutos |
| `ValidationException` sobre o modelo | Região errada: confirme que está usando `us-east-1` |

# Ao final do workshop

Access keys de longo prazo são um risco se vazarem. Quando terminar:

1. Em **IAM > Users >** seu usuário **> Security credentials**, encontre a access key e use **Actions > Deactivate** e depois **Actions > Delete**
2. Remova o perfil da sua máquina, apagando as seções `[workshop]` de `~/.aws/credentials` e `[profile workshop]` de `~/.aws/config`
3. Se o usuário não for mais usado, apague-o em **IAM > Users**